# ENIGH 2024 Nueva Serie — integración de los 17 conjuntos de datos

**Proyecto:** modelo de estimación de ingresos para población sin historial crediticio
**Fuente:** INEGI, Encuesta Nacional de Ingresos y Gastos de los Hogares 2024, Nueva Serie
**Repositorio de datos:** `Ruhguevara/ENIGH_INEGI_2024`, carpeta `DAT/`

---

## Qué hace este notebook y qué no

La ENIGH **no es una tabla**. Es un modelo relacional con cinco granos distintos:

| Grano | Llave | Tablas |
|---|---|---|
| Vivienda | `folioviv` | `viviendas` |
| Hogar | `folioviv + foliohog` | `hogares`, `concentradohogar`, `gastoshogar`, `erogaciones`, `gastotarjetas` |
| Persona | `+ numren` | `poblacion`, `ingresos`, `ingresos_jcf`, `gastospersona` |
| Trabajo | `+ id_trabajo` | `trabajos`, `noagro`, `agro` |
| Detalle | `+ clave` / `tipoact` / `numprod` | `noagroimportes`, `agroproductos`, `agroconsumo`, `agrogasto` |

Aplanar las 17 tablas en una sola produce un producto cartesiano sin significado
estadístico: una persona con 2 trabajos, 5 claves de ingreso y un hogar con 40
registros de gasto generaría 400 filas para la misma unidad de observación.

Por eso el resultado de este notebook son **dos tablas analíticas** con grano
declarado y verificado, más artefactos de apoyo:

| Salida | Grano | Uso |
|---|---|---|
| `enigh2024_hogar.parquet` | un hogar | Modelado de ingreso del hogar, deciles, calibración |
| `enigh2024_persona.parquet` | un integrante del hogar | **Estimador de ingreso a nivel persona** |
| `enigh2024_ingresos_clave.parquet` | persona × clave de ingreso | Análisis de composición del ingreso |
| `enigh2024_negocios.parquet` | negocio del hogar | Ingreso de trabajadores independientes |
| `enigh2024_diccionario_final.csv` | variable | Documentación del dataset entregado |

Cada join se ejecuta con `unir()`, que reporta filas antes y después, cobertura y
fan-out. Al final hay una sección de **validación contra las cifras publicadas por
el INEGI**: si el promedio ponderado del ingreso corriente no da 77 864 pesos, el
pipeline está mal y no se debe usar el resultado.

## 0. Requisitos

```bash
pip install pandas>=2.0 numpy pyarrow matplotlib
```

El módulo `enigh2024_pipeline.py` debe estar junto a este notebook.

**Memoria.** Los 17 CSV suman 880 MB en disco. `gastoshogar` solo son 579 MB,
el 66 % del total. Este pipeline **no lo carga completo**: `concentradohogar` ya
trae `gasto_mon` y todos sus componentes trimestralizados. Con eso, el pico de RAM
se queda por debajo de 4 GB.

In [10]:
import os, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import enigh2024_pipeline as ep

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------------------------------------------------- CONFIGURACIÓN
# Apunta a la carpeta DAT/ de tu clon local del repositorio.
RUTA_DAT     = Path("DAT/")
RUTA_SALIDA  = Path("./salida_enigh2024")
RUTA_SALIDA.mkdir(exist_ok=True, parents=True)

R = ep.Rutas(RUTA_DAT)
print("DAT   :", R.dat)
print("Salida:", RUTA_SALIDA.resolve())
print("pandas", pd.__version__, "| numpy", np.__version__)

DAT   : /Users/r/Projects/ENIGH_INEGI_2024/DAT
Salida: /Users/r/Projects/ENIGH_INEGI_2024/salida_enigh2024
pandas 2.3.3 | numpy 2.3.5


### Nota sobre Git LFS

Los 194 CSV del repositorio están bajo Git LFS. Un `git clone` normal deja
**punteros de 130 bytes** en lugar de los datos. Antes de correr esto:

```bash
git lfs install
git clone https://github.com/Ruhguevara/ENIGH_INEGI_2024.git
cd ENIGH_INEGI_2024 && git lfs pull
```

`inventario()` detecta los punteros y lo reporta en la columna `puntero_lfs`.

---
## 1. Inventario: ¿está completo lo que necesitamos?

Primer control de calidad. Verifica que existan las 17 carpetas, con sus
subcarpetas `conjunto_de_datos`, `diccionario_de_datos` y `catalogos`, y que los
CSV sean datos reales y no punteros LFS.

In [11]:
inv = ep.inventario(R)
display(inv[["tabla", "nivel", "mb", "datos_ok", "puntero_lfs",
             "diccionario_ok", "n_catalogos", "llaves_teoricas"]])

print(f"\nTablas con datos legibles : {inv.datos_ok.sum()} / 17")
print(f"Punteros LFS sin resolver : {inv.puntero_lfs.sum()}")
print(f"Volumen total en disco    : {inv.mb.sum():,.1f} MB")
print(f"Catálogos disponibles     : {inv.n_catalogos.sum()}")

assert inv.datos_ok.all(), (
    "Faltan datos o hay punteros LFS sin resolver. "
    "Corre `git lfs pull` dentro del repositorio antes de continuar."
)

,tabla,nivel,mb,datos_ok,puntero_lfs,diccionario_ok,n_catalogos,llaves_teoricas
0,gastoshogar,detalle,579.04,True,False,True,11,folioviv + foliohog + clave
1,poblacion,persona,92.76,True,False,True,38,folioviv + foliohog + numren
2,concentradohogar,hogar,45.59,True,False,True,6,folioviv + foliohog
3,ingresos,detalle,35.23,True,False,True,3,folioviv + foliohog + numren + clave
4,gastospersona,detalle,32.40,True,False,True,8,folioviv + foliohog + numren + clave
5,hogares,hogar,25.48,True,False,True,9,folioviv + foliohog
6,trabajos,trabajo,20.10,True,False,True,13,folioviv + foliohog + numren + id_trabajo
7,viviendas,vivienda,16.10,True,False,True,26,folioviv
8,noagroimportes,detalle,9.75,True,False,True,3,folioviv + foliohog + numren + id_trabajo + clave
9,noagro,trabajo,6.84,True,False,True,12,folioviv + foliohog + numren + id_trabajo



Tablas con datos legibles : 17 / 17
Punteros LFS sin resolver : 0
Volumen total en disco    : 879.7 MB
Catálogos disponibles     : 159


### Hallazgo 1 — la distribución del peso está muy sesgada

`gastoshogar` concentra dos tercios del volumen. Le siguen `poblacion` (93 MB) y
`concentradohogar` (46 MB). Las 14 tablas restantes juntas no llegan a 180 MB.

Para el estimador de ingresos, **las tablas caras no son las importantes**:
`concentradohogar` (46 MB) sustituye a `gastoshogar` (579 MB) para todo lo que no
sea análisis de gasto por clave de producto.

---
## 2. Esquema: diccionarios de datos consolidados

El tipado no se decide a mano. Cada carpeta trae su `diccionario_de_datos` con el
nemónico, la longitud y el tipo (`C` carácter / `N` numérico) de cada variable.
Consolidarlos da el esquema completo de la base y sirve para documentar el
producto final.

In [12]:
esquema = pd.concat(
    [ep.leer_diccionario(R, t) for t in ep.TABLAS],
    ignore_index=True
)[["tabla", "variable", "etiqueta", "tipo", "longitud", "catalogo"]]

print(f"Variables documentadas en total : {len(esquema):,}")
print(f"Variables únicas por nombre     : {esquema.variable.nunique():,}")
display(esquema.groupby("tabla").agg(
    n_variables=("variable", "size"),
    n_numericas=("tipo", lambda s: (s == "N").sum()),
    n_caracter=("tipo", lambda s: (s == "C").sum()),
).sort_values("n_variables", ascending=False))

Variables documentadas en total : 957
Variables únicas por nombre     : 768


,n_variables,n_numericas,n_caracter
tabla,,,
poblacion,185,27,158
hogares,148,52,96
concentradohogar,126,116,10
noagro,115,68,47
viviendas,82,16,65
agro,66,37,29
trabajos,60,2,58
gastoshogar,31,12,19
agroproductos,25,6,19


In [13]:
# Variables que aparecen en varias tablas: son las candidatas a colisión en los
# joins. Hay que decidir explícitamente cuál se conserva.
compartidas = (esquema.groupby("variable")
                      .agg(n_tablas=("tabla", "nunique"),
                           tablas=("tabla", lambda s: ", ".join(sorted(s))))
                      .query("n_tablas > 1")
                      .sort_values("n_tablas", ascending=False))
display(compartidas.head(25))

,n_tablas,tablas
variable,,
folioviv,17,"agro, agroconsumo, agrogasto, agroproductos, c..."
foliohog,16,"agro, agroconsumo, agrogasto, agroproductos, c..."
numren,11,"agro, agroconsumo, agrogasto, agroproductos, g..."
est_dis,8,"concentradohogar, gastoshogar, gastospersona, ..."
clave,8,"agrogasto, erogaciones, gastoshogar, gastosper..."
upm,8,"concentradohogar, gastoshogar, gastospersona, ..."
factor,8,"concentradohogar, gastoshogar, gastospersona, ..."
id_trabajo,7,"agro, agroconsumo, agrogasto, agroproductos, n..."
entidad,6,"gastoshogar, gastospersona, hogares, ingresos,..."


### Hallazgo 2 — `factor`, `est_dis`, `upm` y `entidad` están replicadas

El INEGI las duplicó en `hogares`, `poblacion`, `ingresos`, `trabajos`,
`gastoshogar` y `gastospersona` para que el usuario no tenga que unir con
`viviendas` solo para poder estimar varianzas.

Consecuencia práctica: **en cada join hay que eliminar la copia de la tabla
derecha**, o pandas creará `factor_dup`. Más abajo se hace de forma sistemática.

---
## 3. Catálogos

Los catálogos son específicos de cada tabla: `catalogos/` de `viviendas` tiene 26
archivos, el de `poblacion` tiene 38, el de `agrogasto` tiene 3. Ningún catálogo
es intercambiable entre carpetas aunque compartan nombre — `si_no.csv` de
`trabajos` y `si_no.csv` de `hogares` pueden diferir.

Por eso se cargan por tabla y se aplican por tabla.

In [14]:
CATALOGOS = {t: ep.leer_catalogos(R, t) for t in ep.TABLAS}

resumen_cat = pd.DataFrame([
    {"tabla": t, "n_catalogos": len(c), "catalogos": ", ".join(sorted(c))[:110]}
    for t, c in CATALOGOS.items()
]).sort_values("n_catalogos", ascending=False)
display(resumen_cat)

# ubica_geo es el catálogo geográfico: entidad + municipio con nombres
cat_geo = CATALOGOS["concentradohogar"]["ubica_geo"]
print(f"\nCatálogo ubica_geo: {len(cat_geo):,} claves municipales")
display(cat_geo.head())

,tabla,n_catalogos,catalogos
6,poblacion,38,"act_pnea, antec_esc, ct_futuro, disc, edo_cony..."
0,viviendas,26,"ab_agua, agua_ent, agua_noe, combus, disp_elec..."
10,trabajos,13,"clas_emp, entidad, id_trabajo, medtrab, no_ing..."
11,noagro,12,"fpago, id_trabajo, lugact, mes, nofpago, numes..."
3,gastoshogar,11,"cantidades, entidad, fecha, forma_pag, frecuen..."
13,agro,9,"fpago, id_trabajo, mes, nofpago, nvo_act, nvo_..."
1,hogares,9,"acc_alim18, entidad, fenomeno, frec_dicon, hab..."
9,gastospersona,8,"cantidades, entidad, forma_pag, frec_rem, gast..."
14,agroproductos,7,"causa_no_cosecha, cicloagr, id_trabajo, produc..."
2,concentradohogar,6,"clase_hog, educa_jefe, est_socio, sexo, tam_lo..."



Catálogo ubica_geo: 1,113 claves municipales


,clave,descripcion
0,ubica_geo,entidad
1,01001,01
2,01002,01
3,01003,01
4,01005,01


---
## 4. Carga tipada de las tablas núcleo

`leer_tabla()` hace tres cosas que `pd.read_csv()` no hace bien por defecto:

1. Prueba UTF-8, UTF-8-BOM y Latin-1 en ese orden.
2. Lee **todo como texto** y luego castea según el diccionario, forzando siempre a
   `str` las llaves. Sin esto, `folioviv = "0100012801"` se convierte en
   `100012801` y **todos los joins con entidades 01-09 fallan en silencio**.
3. Convierte `&` (no especificado del INEGI) a `NaN` en variables carácter.

In [15]:
print("Cargando tablas núcleo:\n")
viviendas   = ep.leer_tabla(R, "viviendas")
hogares     = ep.leer_tabla(R, "hogares")
concentrado = ep.leer_tabla(R, "concentradohogar")
poblacion   = ep.leer_tabla(R, "poblacion")
trabajos    = ep.leer_tabla(R, "trabajos")
ingresos    = ep.leer_tabla(R, "ingresos")

print()
display(ep.resumen_memoria({
    "viviendas": viviendas, "hogares": hogares, "concentrado": concentrado,
    "poblacion": poblacion, "trabajos": trabajos, "ingresos": ingresos,
}))

Cargando tablas núcleo:

  viviendas             90,324 filas x  82 cols   [16 num / 66 car]      16.1 MB en disco     287.6 MB en RAM
  hogares               91,414 filas x 148 cols   [52 num / 96 car]      25.5 MB en disco     395.5 MB en RAM
  concentradohogar      91,414 filas x 126 cols   [116 num / 10 car]      45.6 MB en disco     132.6 MB en RAM
  poblacion            308,598 filas x 185 cols   [27 num / 158 car]      92.8 MB en disco    1934.0 MB en RAM
  trabajos             164,325 filas x  60 cols   [2 num / 58 car]      20.1 MB en disco     395.1 MB en RAM
  ingresos             391,563 filas x  21 cols   [8 num / 13 car]      35.2 MB en disco     290.5 MB en RAM



,objeto,filas,columnas,mb_ram
0,poblacion,"308,598.00",185.00,"1,934.00"
1,hogares,"91,414.00",148.00,395.50
2,trabajos,"164,325.00",60.00,395.10
3,ingresos,"391,563.00",21.00,290.50
4,viviendas,"90,324.00",82.00,287.60
5,concentrado,"91,414.00",126.00,132.60
6,TOTAL,NaN,NaN,"3,435.30"


In [16]:
# Verificación crítica de los ceros a la izquierda
muestra = viviendas.folioviv.head(3).tolist()
print("folioviv de ejemplo:", muestra)
assert viviendas.folioviv.str.len().eq(10).all(), "folioviv perdió longitud fija de 10"
assert concentrado.ubica_geo.str.len().eq(5).all(), "ubica_geo debe tener 5 caracteres"
print("Longitudes de llave correctas: folioviv=10, ubica_geo=5")

# Entidades presentes: si faltan las 01-09 hay un problema de tipado
print("\nEntidades en concentradohogar:",
      sorted(concentrado.ubica_geo.str[:2].unique())[:12], "...")
print("Total de entidades:", concentrado.ubica_geo.str[:2].nunique(), "(esperado 32)")

folioviv de ejemplo: ['0100001901', '0100001902', '0100001904']
Longitudes de llave correctas: folioviv=10, ubica_geo=5

Entidades en concentradohogar: ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'] ...
Total de entidades: 32 (esperado 32)


---
## 5. Verificación del grano

Antes de unir nada hay que confirmar el grano real de cada tabla contra el grano
teórico del modelo entidad-relación. Un join `1:1` sobre una tabla que en realidad
es `1:N` multiplica filas sin avisar.

In [17]:
print("Grano teórico vs. grano observado:\n")
granos = {}
for nombre, df in [("viviendas", viviendas), ("hogares", hogares),
                   ("concentradohogar", concentrado), ("poblacion", poblacion),
                   ("trabajos", trabajos), ("ingresos", ingresos)]:
    granos[nombre] = ep.validar_llave(df, list(ep.TABLAS[nombre].llaves), nombre)

print("\nGrano de trabajos a nivel PERSONA (esperamos que NO sea único: multiempleo):")
ep.validar_llave(trabajos, ["folioviv", "foliohog", "numren"], "trabajos@persona")

print("\nGrano de ingresos a nivel PERSONA (esperamos que NO sea único: varias claves):")
ep.validar_llave(ingresos, ["folioviv", "foliohog", "numren"], "ingresos@persona")

Grano teórico vs. grano observado:

  [ok] viviendas: grano único en (folioviv) — 90,324 filas
  [ok] hogares: grano único en (folioviv + foliohog) — 91,414 filas
  [ok] concentradohogar: grano único en (folioviv + foliohog) — 91,414 filas
  [ok] poblacion: grano único en (folioviv + foliohog + numren) — 308,598 filas
  [ok] trabajos: grano único en (folioviv + foliohog + numren + id_trabajo) — 164,325 filas
  [ok] ingresos: grano único en (folioviv + foliohog + numren + clave) — 391,563 filas

Grano de trabajos a nivel PERSONA (esperamos que NO sea único: multiempleo):
  [!] trabajos@persona: 13,943 filas duplicadas en (folioviv + foliohog + numren) → la tabla es de grano más fino de lo esperado, NO unir 1:1

Grano de ingresos a nivel PERSONA (esperamos que NO sea único: varias claves):
  [!] ingresos@persona: 189,198 filas duplicadas en (folioviv + foliohog + numren) → la tabla es de grano más fino de lo esperado, NO unir 1:1


False

In [18]:
# Conteos de referencia del universo muestral
n_viv = viviendas.folioviv.nunique()
n_hog = len(concentrado)
n_per = len(poblacion)

print(f"Viviendas en muestra : {n_viv:,}")
print(f"Hogares              : {n_hog:,}   ({n_hog/n_viv:.3f} hogares por vivienda)")
print(f"Personas             : {n_per:,}   ({n_per/n_hog:.2f} personas por hogar)")
print(f"\nMuestra efectiva publicada por el INEGI (Diseño muestral): 105 718 viviendas")
print(f"Diferencia: {n_viv - 105_718:+,}")

Viviendas en muestra : 90,324
Hogares              : 91,414   (1.012 hogares por vivienda)
Personas             : 308,598   (3.38 personas por hogar)

Muestra efectiva publicada por el INEGI (Diseño muestral): 105 718 viviendas
Diferencia: -15,394


### Hallazgo 3 — hogares adicionales

El número de hogares supera al de viviendas porque en algunas viviendas hay más de
un hogar (`foliohog` 2 a 5). El diseño muestral asumió 1.02 hogares por vivienda;
el dato observado confirma el orden de magnitud.

Esto importa: **la unidad de observación es el hogar, no la vivienda**. Si tu
enriquecimiento geográfico opera sobre domicilios, el mapeo natural es
vivienda → domicilio, y varios hogares pueden compartir uno.

---
## 6. Tabla analítica de HOGAR

Composición: `concentradohogar` como base (ya trae ingresos y gastos construidos y
trimestralizados) + características de la vivienda + equipamiento del hogar.

Grano de salida: **un hogar**. `concentradohogar` es la base y no la derecha de
ningún join, así que el conteo de filas no debe cambiar en ninguna etapa.

In [19]:
def sin_colisiones(der, izq, llaves):
    # Elimina de la derecha las columnas ya presentes en la izquierda, salvo las llaves.
    dup = [c for c in der.columns if c in izq.columns and c not in llaves]
    if dup:
        print(f"      quitando de la derecha {len(dup)} columnas ya presentes: {dup[:8]}"
              f"{'...' if len(dup) > 8 else ''}")
    return der.drop(columns=dup)


print("Construcción de la tabla HOGAR\n")
hogar = concentrado.copy()
n0 = len(hogar)

hogar = ep.unir(hogar, sin_colisiones(viviendas, hogar, ["folioviv"]),
                on=["folioviv"], how="left", validar="m:1",
                nombre="concentradohogar + viviendas")

hogar = ep.unir(hogar, sin_colisiones(hogares, hogar, ["folioviv", "foliohog"]),
                on=["folioviv", "foliohog"], how="left", validar="1:1",
                nombre="+ hogares")

assert len(hogar) == n0, f"El grano de hogar cambió: {n0:,} -> {len(hogar):,}"
print(f"\nGrano preservado: {len(hogar):,} hogares, {hogar.shape[1]} columnas")

Construcción de la tabla HOGAR

      quitando de la derecha 7 columnas ya presentes: ['combus', 'ubica_geo', 'tam_loc', 'est_socio', 'est_dis', 'upm', 'factor']
  concentradohogar + viviendas   izq    91,414 x126  + der    90,324 =    91,414 x200  | +74 cols | pareja 100.0% (0 sin match)
      quitando de la derecha 3 columnas ya presentes: ['est_dis', 'upm', 'factor']
  + hogares                      izq    91,414 x200  + der    91,414 =    91,414 x343  | +143 cols | pareja 100.0% (0 sin match)

Grano preservado: 91,414 hogares, 343 columnas


In [20]:
# Geografía: desglose de ubica_geo y etiquetado
hogar = ep.desglosar_ubica_geo(hogar)
hogar = ep.etiquetar(hogar, CATALOGOS["concentradohogar"], {
    "ubica_geo": "ubica_geo",
    "tam_loc":   "tam_loc",
    "est_socio": "est_socio",
    "clase_hog": "clase_hog",
    "sexo_jefe": "sexo",
    "educa_jefe": "educa_jefe",
})

# Dominio de estudio del diseño muestral: urbano = localidades de 2 500 y más
hogar["ambito"] = np.where(hogar["tam_loc"] == "4", "Rural", "Urbano")

# Ingreso expresado en salarios mínimos mensuales (smg viene en la propia tabla)
hogar["ing_cor_sm"] = hogar["ing_cor"] / (hogar["smg"] * 3)
hogar["ing_cor_pc"] = hogar["ing_cor"] / hogar["tot_integ"]

display(hogar[["folioviv", "foliohog", "cve_ent", "cve_mun", "ubica_geo_desc",
               "ambito", "est_socio_desc", "factor", "tot_integ",
               "ing_cor", "ing_cor_sm", "ingtrab", "gasto_mon"]].head(8))

,folioviv,foliohog,cve_ent,cve_mun,ubica_geo_desc,ambito,est_socio_desc,factor,tot_integ,ing_cor,ing_cor_sm,ingtrab,gasto_mon
0,0100001901,1,01,001,01,Urbano,Medio alto,207,4,"138,232.38",2.06,"130,518.10","47,478.66"
1,0100001902,1,01,001,01,Urbano,Medio alto,207,4,"118,014.04",1.76,"103,829.72","38,782.74"
2,0100001904,1,01,001,01,Urbano,Medio alto,207,2,"46,866.32",0.70,"45,580.61","28,601.26"
3,0100001905,1,01,001,01,Urbano,Medio alto,207,4,"110,430.10",1.64,"97,169.95","43,509.83"
4,0100002501,1,01,001,01,Urbano,Medio bajo,196,4,"99,494.12",1.48,"93,687.67","132,552.40"
5,0100002502,1,01,001,01,Urbano,Medio bajo,196,4,"119,559.08",1.78,"110,484.26","74,443.10"
6,0100002504,1,01,001,01,Urbano,Medio bajo,196,4,"78,830.79",1.17,"67,110.07","35,896.53"
7,0100002505,1,01,001,01,Urbano,Medio bajo,196,4,"41,394.62",0.62,"38,201.08","148,492.70"


### Hallazgo 4 — `smg` viene dentro de `concentradohogar`

No hace falta traer el salario mínimo de fuera ni deflactar para expresar el
objetivo en múltiplos de salario mínimo. `ing_cor / (smg * 3)` da el ingreso
corriente trimestral del hogar en salarios mínimos mensuales, y es directamente
comparable entre ediciones 2016-2024 sin índice de precios.

Esto conecta con la definición de bandera objetivo del modelo.

---
## 7. Ingresos: de formato largo a una fila por persona

`ingresos` es una tabla larga: una fila por persona y clave de ingreso. Para
modelar hay que pivotearla, pero pivotear por las ~180 claves individuales genera
un DataFrame disperso e inmanejable.

La alternativa es agregar por **rubro de la Nueva Serie**. Para no depender de
rangos de claves que cambian entre ediciones, la clasificación se deriva de la
*descripción* del catálogo `ingresos_cat.csv` mediante reglas de texto.

**Esta clasificación hay que revisarla.** La celda siguiente imprime el mapeo
completo y las claves que ninguna regla logró clasificar.

In [21]:
cat_ingresos = CATALOGOS["ingresos"]["ingresos_cat"]
clasif = ep.clasificar_claves_ingreso(cat_ingresos)

print("Distribución de claves por rubro:")
display(clasif.rubro.value_counts().to_frame("n_claves"))

sin_clasificar = clasif.query("rubro == 'sin_clasificar'")
if len(sin_clasificar):
    print(f"\n[!] {len(sin_clasificar)} claves sin clasificar — REVISAR Y AJUSTAR "
          "ep.REGLAS_RUBRO_INGRESO antes de usar el desglose por rubro:")
    display(sin_clasificar)
else:
    print("\nTodas las claves quedaron clasificadas.")

Distribución de claves por rubro:


,n_claves
rubro,
transferencias,19
sin_clasificar,14
trabajo_independiente,14
percepciones_financieras,13
renta_propiedad,12
trabajo_subordinado,10
otros_trabajos,1



[!] 14 claves sin clasificar — REVISAR Y AJUSTAR ep.REGLAS_RUBRO_INGRESO antes de usar el desglose por rubro:


,clave,descripcion,rubro
3,P004,Horas extras,sin_clasificar
11,P013,Otros ingresos del trabajo principal,sin_clasificar
12,P014,Monto recibido en el trabajo secundario,sin_clasificar
17,P020,Otros ingresos del trabajo secundario,sin_clasificar
19,P022,Total de ingresos de trabajos realizados en lo...,sin_clasificar
26,P029,Rendimientos provenientes de bonos o cédulas,sin_clasificar
42,P103,Jóvenes Escribiendo el Futuro (Educación Super...,sin_clasificar
50,P049,Total de ingresos no considerados en los anter...,sin_clasificar
51,P050,Ingresos anuales por rendimientos de acciones ...,sin_clasificar
55,P054,"Venta de monedas, metales preciosos, joyas y o...",sin_clasificar


In [22]:
# Mapeo completo, para auditoría
with pd.option_context("display.max_rows", 250):
    display(clasif.sort_values(["rubro", "clave"]))

,clave,descripcion,rubro
18,P021,Total de ingresos de otros trabajos realizados...,otros_trabajos
25,P028,Intereses provenientes de préstamos a terceros,percepciones_financieras
52,P051,"Retiro de inversiones, ahorros, tandas, cajas ...",percepciones_financieras
53,P052,Pagos recibidos de préstamos que usted hizo a ...,percepciones_financieras
54,P053,Préstamos recibidos de personas ajenas al hoga...,percepciones_financieras
58,P057,"Herencias, dotes y legados",percepciones_financieras
59,P058,Loterías y juegos de azar,percepciones_financieras
60,P059,"Venta de casas, condominios, etcétera, que est...",percepciones_financieras
61,P060,Venta de terrenos que están dentro y fuera del...,percepciones_financieras
62,P061,"Venta de maquinaria, equipos, animales de prod...",percepciones_financieras


In [23]:
ing_persona = ep.agregar_ingresos_persona(ingresos, clasif)
ep.validar_llave(ing_persona, ["folioviv", "foliohog", "numren"], "ingresos por persona")

print(f"\nRegistros de ingreso (largo) : {len(ingresos):,}")
print(f"Personas con ingreso (ancho) : {len(ing_persona):,}")
print(f"Factor de compresión         : {len(ingresos)/len(ing_persona):.2f} claves por perceptor")
display(ing_persona.head())

  [ok] ingresos por persona: grano único en (folioviv + foliohog + numren) — 202,365 filas

Registros de ingreso (largo) : 391,563
Personas con ingreso (ancho) : 202,365
Factor de compresión         : 1.93 claves por perceptor


,folioviv,foliohog,numren,ing_tri_total,n_claves_ingreso,ing_tri_otros_trabajos,ing_tri_percepciones_financieras,ing_tri_renta_propiedad,ing_tri_sin_clasificar,ing_tri_trabajo_independiente,ing_tri_trabajo_subordinado,ing_tri_transferencias
0,0100001901,1,01,"78,789.11",5,0.00,0.00,"4,402.17","17,608.69",0.00,"56,778.25",0.00
1,0100001901,1,02,"31,157.59",3,0.00,0.00,0.00,586.95,0.00,"30,570.64",0.00
2,0100001902,1,01,"56,739.12",3,0.00,0.00,"7,826.08",0.00,0.00,"48,913.04",0.00
3,0100001902,1,03,"29,347.82",1,0.00,0.00,0.00,0.00,0.00,"29,347.82",0.00
4,0100001904,1,01,"11,861.40",3,0.00,0.00,0.00,"5,869.56",0.00,"5,991.84",0.00


### Reconciliación contra `concentradohogar`

Prueba de que el pivote no perdió ni duplicó dinero: la suma de los ingresos
corrientes de todas las personas de un hogar debe reproducir `ing_cor`, salvo por
dos componentes que **no viven en la tabla `ingresos`**:

- la estimación del alquiler de la vivienda propia (`estim_alqu`), que es imputada
  a nivel hogar;
- los ingresos no monetarios de los negocios, que entran vía `agro` / `noagro`.

Por eso la reconciliación se hace contra `ing_cor - estim_alqu`.

In [24]:
rubros_corrientes = [c for c in ing_persona.columns
                     if c.startswith("ing_tri_")
                     and c not in ("ing_tri_total", "ing_tri_percepciones_financieras",
                                   "ing_tri_sin_clasificar")]
print("Rubros considerados corrientes:", rubros_corrientes)

ing_hogar = (ing_persona.assign(ing_corriente_personas=lambda d: d[rubros_corrientes].sum(axis=1))
             .groupby(["folioviv", "foliohog"], as_index=False)
             .agg(ing_corriente_personas=("ing_corriente_personas", "sum"),
                  perc_fin_personas=("ing_tri_percepciones_financieras", "sum"),
                  n_perceptores_calc=("ing_tri_total", "size")))

rec = hogar[["folioviv", "foliohog", "ing_cor", "estim_alqu", "percep_ing", "factor"]] \
        .merge(ing_hogar, on=["folioviv", "foliohog"], how="left")
rec["esperado"] = rec["ing_cor"] - rec["estim_alqu"]
rec["dif"] = rec["ing_corriente_personas"].fillna(0) - rec["esperado"]
rec["dif_rel"] = rec["dif"] / rec["esperado"].replace(0, np.nan)

print(f"\nHogares con diferencia < 1%   : {(rec.dif_rel.abs() < 0.01).mean():.1%}")
print(f"Diferencia mediana absoluta   : {rec.dif.abs().median():,.0f} pesos")
print(f"Diferencia total ponderada    : "
      f"{(rec.dif * rec.factor).sum() / (rec.esperado * rec.factor).sum():.2%}")
display(rec.dif_rel.describe(percentiles=[.05, .25, .5, .75, .95]).to_frame())

Rubros considerados corrientes: ['ing_tri_otros_trabajos', 'ing_tri_renta_propiedad', 'ing_tri_trabajo_independiente', 'ing_tri_trabajo_subordinado', 'ing_tri_transferencias']

Hogares con diferencia < 1%   : 41.2%
Diferencia mediana absoluta   : 1,508 pesos
Diferencia total ponderada    : -9.36%


,dif_rel
count,"91,351.00"
mean,-0.12
std,0.18
min,-1.00
5%,-0.51
25%,-0.16
50%,-0.03
75%,0.00
95%,0.00
max,1.80


**Cómo leer este resultado.** Una diferencia sistemática y grande significa que la
clasificación de claves por rubro está mal, o que faltan los ingresos no monetarios
de negocios. Diferencias pequeñas y dispersas son normales: `concentradohogar`
aplica ajustes de validación que no se reflejan clave por clave.

Si la diferencia ponderada supera el 2 %, **usa `concentradohogar` como autoridad**
para los agregados de hogar y el desglose por rubro solo como covariable
descriptiva a nivel persona.

---
## 8. Trabajos: trabajo principal y multiempleo

`trabajos` tiene grano persona × `id_trabajo`. Para el modelo se necesita una fila
por persona, así que se separa en dos piezas: el trabajo principal (el de menor
`id_trabajo`) y un resumen de multiempleo.

In [25]:
trabajo_ppal, multiempleo = ep.seleccionar_trabajo_principal(trabajos)

ep.validar_llave(trabajo_ppal, ["folioviv", "foliohog", "numren"], "trabajo principal")
print(f"\nRegistros de trabajo    : {len(trabajos):,}")
print(f"Personas con trabajo    : {len(trabajo_ppal):,}")
print(f"Con más de un trabajo   : {multiempleo.multiempleo.sum():,} "
      f"({multiempleo.multiempleo.mean():.1%})")

# Etiquetado ocupacional: SINCO, SCIAN, tamaño de empresa. Este es el puente
# hacia DENUE (SCIAN) y hacia IMSS (tamaño de empresa / registro patronal).
trabajo_ppal = ep.etiquetar(trabajo_ppal, CATALOGOS["trabajos"], {
    "sinco": "sinco", "scian": "scian", "tam_emp": "tam_emp",
    "clas_emp": "clas_emp", "subor": "si_no", "indep": "si_no",
    "contrato": "si_no", "no_ing": "no_ing",
})
display(trabajo_ppal[["folioviv", "numren", "id_trabajo", "subor", "htrab",
                      "scian", "scian_desc", "tam_emp_desc"]].head(8))

  [ok] trabajo principal: grano único en (folioviv + foliohog + numren) — 150,382 filas

Registros de trabajo    : 164,325
Personas con trabajo    : 150,382
Con más de un trabajo   : 13,943 (9.3%)


,folioviv,numren,id_trabajo,subor,htrab,scian,scian_desc,tam_emp_desc
0,0100001901,01,1,1,48,3360,Fabricación de equipo de transporte,No sabe
1,0100001901,02,1,1,45,8121,Servicios personales,De 2 a 5 personas
2,0100001902,01,1,1,48,3360,Fabricación de equipo de transporte,De 501 a más personas
3,0100001902,03,1,1,48,3360,Fabricación de equipo de transporte,De 501 a más personas
4,0100001904,01,1,1,12,4611,"Comercio al por menor de abarrotes, alimentos,...",De 2 a 5 personas
5,0100001904,02,1,1,40,8112,Servicios de reparación y mantenimiento de equ...,De 2 a 5 personas
6,0100001905,01,1,1,48,3270,Fabricación de productos a base de minerales n...,De 501 a más personas
7,0100001905,02,1,1,10,4651,Comercio al por menor de artículos de papelerí...,De 2 a 5 personas


In [26]:
# Prefijo a las columnas de trabajo para que no colisionen al integrar
llaves_p = ["folioviv", "foliohog", "numren"]
trabajo_ppal = trabajo_ppal.rename(columns={
    c: f"trab_{c}" for c in trabajo_ppal.columns
    if c not in llaves_p and c not in ("entidad", "est_dis", "upm", "factor")
})
trabajo_ppal = trabajo_ppal.drop(columns=[c for c in ("entidad", "est_dis", "upm", "factor")
                                          if c in trabajo_ppal.columns])
print("Columnas de trabajo listas:", [c for c in trabajo_ppal.columns if c.startswith("trab_")][:12], "...")

Columnas de trabajo listas: ['trab_id_trabajo', 'trab_trapais', 'trab_subor', 'trab_indep', 'trab_personal', 'trab_pago', 'trab_contrato', 'trab_tipocontr', 'trab_pres_1', 'trab_pres_2', 'trab_pres_3', 'trab_pres_4'] ...


### Hallazgo 5 — `trabajos` trae SCIAN y SINCO a cuatro dígitos

Esta es probablemente la variable más valiosa del microdato para tu pipeline.
`scian` a 4 dígitos es la **misma clasificación que usa el DENUE**, y `tam_emp` es
la misma dimensión que puedes derivar del registro patronal del IMSS.

Con eso se puede estimar `P(ingreso | SCIAN, tam_emp, entidad, tam_loc, est_socio)`
en la ENIGH y trasladarla al solicitante mediante el match geoespacial que ya
tienes con DENUE. Es la vía para convertir "hay un establecimiento de tal giro
cerca" en una expectativa de ingreso con fundamento estadístico.

---
## 9. Tabla analítica de PERSONA

Grano de salida: **un integrante del hogar**. Composición:

```
poblacion  (base, grano persona)
   + trabajo principal          (m:1, solo ocupados)
   + resumen de multiempleo     (m:1)
   + ingresos agregados         (m:1, solo perceptores)
   + contexto de hogar/vivienda (m:1, se replica a todos los integrantes)
```

Las columnas heredadas del hogar se prefijan con `hog_` para que quede explícito
que **no son atributos de la persona**: replicar el ingreso del hogar en cada
integrante y luego modelarlo como si fuera individual es un error frecuente.

In [27]:
print("Construcción de la tabla PERSONA\n")
persona = poblacion.copy()
n0 = len(persona)

persona = ep.unir(persona, trabajo_ppal, on=llaves_p, how="left", validar="1:1",
                  nombre="poblacion + trabajo principal")
persona = ep.unir(persona, multiempleo, on=llaves_p, how="left", validar="1:1",
                  nombre="+ multiempleo")
persona = ep.unir(persona, ing_persona, on=llaves_p, how="left", validar="1:1",
                  nombre="+ ingresos agregados")

assert len(persona) == n0, f"El grano de persona cambió: {n0:,} -> {len(persona):,}"
print(f"\nGrano preservado: {len(persona):,} personas, {persona.shape[1]} columnas")

Construcción de la tabla PERSONA

  poblacion + trabajo principal  izq   308,598 x185  + der   150,382 =   308,598 x246  | +61 cols | pareja  48.7% (158,216 sin match)
  + multiempleo                  izq   308,598 x246  + der   150,382 =   308,598 x248  | +2 cols | pareja  48.7% (158,216 sin match)
  + ingresos agregados           izq   308,598 x248  + der   202,365 =   308,598 x257  | +9 cols | pareja  65.6% (106,233 sin match)

Grano preservado: 308,598 personas, 257 columnas


In [28]:
# Contexto del hogar y de la vivienda, prefijado
cols_hogar = [
    "folioviv", "foliohog", "ubica_geo", "cve_ent", "cve_mun", "ubica_geo_desc",
    "tam_loc", "tam_loc_desc", "est_socio", "est_socio_desc", "ambito",
    "est_dis", "upm", "factor", "clase_hog_desc", "sexo_jefe_desc", "educa_jefe",
    "tot_integ", "percep_ing", "ocupados", "ing_cor", "ing_cor_pc", "ing_cor_sm",
    "ingtrab", "trabajo", "negocio", "rentas", "transfer", "estim_alqu",
    "gasto_mon", "smg", "tenencia", "num_cuarto", "mat_pisos", "disp_agua",
    "drenaje", "disp_elect",
]
cols_hogar = [c for c in cols_hogar if c in hogar.columns]
ctx = hogar[cols_hogar].rename(columns={
    c: f"hog_{c}" for c in cols_hogar
    if c not in ("folioviv", "foliohog", "est_dis", "upm", "factor")
})

# factor ya viene en poblacion (mismo valor); nos quedamos con el de poblacion
ctx = ctx.drop(columns=[c for c in ("est_dis", "upm", "factor") if c in persona.columns
                        and c in ctx.columns])

persona = ep.unir(persona, ctx, on=["folioviv", "foliohog"], how="left", validar="m:1",
                  nombre="+ contexto hogar/vivienda")
assert len(persona) == n0
print(f"\nTabla PERSONA final: {len(persona):,} filas x {persona.shape[1]} columnas")

  + contexto hogar/vivienda      izq   308,598 x257  + der    91,414 =   308,598 x288  | +31 cols | pareja 100.0% (0 sin match)

Tabla PERSONA final: 308,598 filas x 288 columnas


In [29]:
# Etiquetado sociodemográfico
persona = ep.etiquetar(persona, CATALOGOS["poblacion"], {
    "sexo": "sexo", "parentesco": "parentesco", "nivelaprob": "nivelaprob",
    "edo_conyug": "edo_conyug", "asis_esc": "si_no", "alfabetism": "alfabetism",
})

# Variables derivadas para el modelo
persona["edad"] = pd.to_numeric(persona["edad"], errors="coerce")
persona["es_perceptor"] = persona["ing_tri_total"].fillna(0) > 0
persona["es_ocupado"] = persona["trab_id_trabajo"].notna()
persona["ing_tri_sm"] = persona["ing_tri_total"] / (persona["hog_smg"] * 3)
persona["grupo_edad"] = pd.cut(
    persona["edad"], [-1, 11, 19, 29, 39, 49, 59, 200],
    labels=["0-11", "12-19", "20-29", "30-39", "40-49", "50-59", "60+"])

print(f"Perceptores de ingreso : {persona.es_perceptor.sum():,} "
      f"({persona.es_perceptor.mean():.1%})")
print(f"Ocupados con trabajo   : {persona.es_ocupado.sum():,} "
      f"({persona.es_ocupado.mean():.1%})")

      [skip] catálogo ausente: alfabetism
Perceptores de ingreso : 202,365 (65.6%)
Ocupados con trabajo   : 150,382 (48.7%)


---
## 10. Negocios del hogar: `agro` y `noagro`

Estas dos tablas reconstruyen el ingreso de los trabajadores independientes, que
es exactamente el segmento que el registro administrativo del IMSS no ve. Se
consolidan en una sola tabla de negocios con grano persona × trabajo.

In [30]:
agro   = ep.leer_tabla(R, "agro")
noagro = ep.leer_tabla(R, "noagro")

llaves_neg = ["folioviv", "foliohog", "numren", "id_trabajo"]
comunes = ["ing_tri", "ero_tri", "ventas_tri", "auto_tri", "otros_tri", "gasto_tri",
           "t_emp", "t_cpago", "reg_cont", "reg_not"]

def normaliza_negocio(df, tipo):
    cols = llaves_neg + [c for c in comunes if c in df.columns]
    out = df[cols].copy()
    out["tipo_negocio"] = tipo
    # agro tiene grano adicional por tipoact: se agrega a nivel trabajo
    num = [c for c in out.columns if out[c].dtype.kind in "if"]
    return out.groupby(llaves_neg + ["tipo_negocio"], as_index=False)[num].sum()

negocios = pd.concat([normaliza_negocio(agro, "agropecuario"),
                      normaliza_negocio(noagro, "no_agropecuario")],
                     ignore_index=True)

print(f"Negocios agropecuarios     : {(negocios.tipo_negocio=='agropecuario').sum():,}")
print(f"Negocios no agropecuarios  : {(negocios.tipo_negocio=='no_agropecuario').sum():,}")
print(f"Personas con negocio       : {negocios[llaves_neg[:3]].drop_duplicates().shape[0]:,}")
display(negocios.groupby("tipo_negocio")[["ing_tri", "ventas_tri", "gasto_tri"]]
                .describe().T.head(12))

  agro                  17,442 filas x  66 cols   [37 num / 29 car]       2.8 MB en disco      27.4 MB en RAM
  noagro                23,109 filas x 115 cols   [68 num / 47 car]       6.8 MB en disco      56.3 MB en RAM
Negocios agropecuarios     : 13,779
Negocios no agropecuarios  : 23,109
Personas con negocio       : 33,944


tipo_negocio      agropecuario  no_agropecuario
ing_tri    count     13,779.00        23,109.00
           mean       6,923.81        19,717.28
           std       29,342.54        31,479.51
           min            0.00             0.00
           25%            0.00         3,914.51
           50%        1,220.86        11,249.99
           75%        4,985.74        25,180.32
           max    2,188,032.78     1,602,652.13
ventas_tri count     13,779.00        23,109.00
           mean      11,740.17        48,450.73
           std       76,884.90        88,108.28
           min            0.00             0.00

In [31]:
# Resumen de negocio a nivel persona, para adjuntar a la tabla PERSONA
neg_persona = (negocios.groupby(llaves_p, as_index=False)
                       .agg(neg_n=("id_trabajo", "size"),
                            neg_ing_tri=("ing_tri", "sum"),
                            neg_ventas_tri=("ventas_tri", "sum"),
                            neg_gasto_tri=("gasto_tri", "sum"),
                            neg_personal=("t_emp", "max")))
neg_persona["neg_tiene"] = True

persona = ep.unir(persona, neg_persona, on=llaves_p, how="left", validar="1:1",
                  nombre="persona + resumen de negocios")
persona["neg_tiene"] = persona["neg_tiene"].fillna(False)
print(f"\nPersonas con negocio propio: {persona.neg_tiene.sum():,}")

  persona + resumen de negocios  izq   308,598 x297  + der    33,944 =   308,598 x303  | +6 cols | pareja  11.0% (274,654 sin match)

Personas con negocio propio: 33,944


---
## 11. Validación contra las cifras publicadas por el INEGI

Esta es la puerta de aceptación del pipeline. Los valores de referencia salen de
la *Presentación de resultados ENIGH 2024* (INEGI, julio 2025), en pesos de 2024.

Si estos números no se reproducen, hay un error de lectura, tipado o ponderación,
y el dataset no debe usarse.

In [32]:
def media_ponderada(df, valor, peso="factor"):
    v = pd.to_numeric(df[valor], errors="coerce")
    w = pd.to_numeric(df[peso], errors="coerce")
    m = v.notna() & w.notna()
    return float(np.average(v[m], weights=w[m]))


REFERENCIA = {
    "ing_cor":    77_864,   # ingreso corriente promedio trimestral por hogar
    "ingtrab":    51_099,   # ingreso por trabajo
    "trabajo":    43_665,   # remuneraciones por trabajo subordinado
    "negocio":     6_057,   # trabajo independiente
    "rentas":      3_834,   # renta de la propiedad
    "transfer":   13_799,   # transferencias
    "estim_alqu":  9_066,   # estimación del alquiler de la vivienda
    "tot_integ":    3.35,   # tamaño del hogar
    "percep_ing":   2.20,   # perceptores por hogar
    "ocupados":     1.63,   # ocupados por hogar
}

val = []
for var, esperado in REFERENCIA.items():
    if var not in hogar.columns:
        val.append({"variable": var, "publicado": esperado, "calculado": np.nan,
                    "dif_%": np.nan, "ok": False}); continue
    calc = media_ponderada(hogar, var)
    val.append({"variable": var, "publicado": esperado, "calculado": round(calc, 2),
                "dif_%": round(100 * (calc - esperado) / esperado, 2),
                "ok": abs(calc - esperado) / esperado < 0.01})

validacion = pd.DataFrame(val)
display(validacion)

print(f"\nHogares expandidos : {hogar.factor.sum():,.0f}")
print(f"Personas expandidas: {persona.factor.sum():,.0f}")
print(f"\nPruebas superadas  : {validacion.ok.sum()} / {len(validacion)}")
if not validacion.ok.all():
    print("[!] Revisa el tipado y la ponderación antes de continuar.")

,variable,publicado,calculado,dif_%,ok
0,ing_cor,"77,864.00","77,863.84",-0.00,True
1,ingtrab,"51,099.00","51,099.27",0.00,True
2,trabajo,"43,665.00","43,664.66",-0.00,True
3,negocio,"6,057.00","6,056.77",-0.00,True
4,rentas,"3,834.00","3,833.84",-0.00,True
5,transfer,"13,799.00","13,799.37",0.00,True
6,estim_alqu,"9,066.00","9,066.09",0.00,True
7,tot_integ,3.35,3.35,0.11,True
8,percep_ing,2.20,2.20,0.10,True
9,ocupados,1.63,1.64,0.60,True



Hogares expandidos : 38,830,230
Personas expandidas: 130,325,969

Pruebas superadas  : 10 / 10


In [33]:
# Urbano / rural: referencia publicada 85 550 urbano y 48 004 rural
amb = (hogar.groupby("ambito")
            .apply(lambda g: pd.Series({
                "hogares_muestra": len(g),
                "hogares_expandidos": g.factor.sum(),
                "ing_cor_promedio": media_ponderada(g, "ing_cor"),
            }), include_groups=False))
amb["publicado"] = amb.index.map({"Urbano": 85_550, "Rural": 48_004})
amb["dif_%"] = 100 * (amb.ing_cor_promedio - amb.publicado) / amb.publicado
display(amb)

,hogares_muestra,hogares_expandidos,ing_cor_promedio,publicado,dif_%
ambito,,,,,
Rural,"35,825.00","7,948,646.00","48,004.00",48004,0.00
Urbano,"55,589.00","30,881,584.00","85,549.50",85550,-0.00


In [34]:
def deciles_ponderados(df, ingreso="ing_cor", peso="factor", n=10):
    # Deciles al estilo INEGI: el hogar que cruza el umbral se reparte entre dos
    # deciles en lugar de asignarse entero a uno, tal como en el documento
    # "Descripcion del calculo de los principales indicadores".
    d = df[[ingreso, peso]].dropna().sort_values(ingreso).reset_index(drop=True)
    w = d[peso].to_numpy(float)
    y = d[ingreso].to_numpy(float)
    sup = np.cumsum(w)
    inf = sup - w
    total = sup[-1]
    corte = total / n
    filas = []
    for i in range(n):
        lo, hi = corte * i, corte * (i + 1)
        wi = np.clip(np.minimum(sup, hi) - np.maximum(inf, lo), 0, None)
        filas.append({"decil": i + 1,
                      "hogares_expandidos": wi.sum(),
                      "ing_cor_promedio": np.average(y, weights=wi) if wi.sum() else np.nan})
    return pd.DataFrame(filas)


PUB_DECILES = [16_795, 28_297, 36_845, 45_245, 54_308,
               64_600, 77_451, 95_291, 123_712, 236_095]

dec = deciles_ponderados(hogar)
dec["publicado"] = PUB_DECILES
dec["dif_%"] = 100 * (dec.ing_cor_promedio - dec.publicado) / dec.publicado
display(dec)
print(f"\nRazón decil X / decil I calculada: "
      f"{dec.ing_cor_promedio.iloc[-1]/dec.ing_cor_promedio.iloc[0]:.2f} "
      f"(publicada 14.06)")

,decil,hogares_expandidos,ing_cor_promedio,publicado,dif_%
0,1,"3,883,023.00","16,795.33",16795,0.00
1,2,"3,883,023.00","28,296.82",28297,-0.00
2,3,"3,883,023.00","36,844.61",36845,-0.00
3,4,"3,883,023.00","45,244.52",45245,-0.00
4,5,"3,883,023.00","54,307.66",54308,-0.00
5,6,"3,883,023.00","64,599.75",64600,-0.00
6,7,"3,883,023.00","77,450.79",77451,-0.00
7,8,"3,883,023.00","95,291.21",95291,0.00
8,9,"3,883,023.00","123,712.44",123712,0.00
9,10,"3,883,023.00","236,095.31",236095,0.00



Razón decil X / decil I calculada: 14.06 (publicada 14.06)


In [35]:
def gini_ponderado(y, w):
    y = np.asarray(y, float); w = np.asarray(w, float)
    m = np.isfinite(y) & np.isfinite(w) & (w > 0)
    y, w = y[m], w[m]
    o = np.argsort(y); y, w = y[o], w[o]
    cw = np.cumsum(w)
    cy = np.cumsum(w * y)
    cy = cy / cy[-1]
    cw = cw / cw[-1]
    return float(1 - np.sum((cw[1:] - cw[:-1]) * (cy[1:] + cy[:-1])))


g_con = gini_ponderado(hogar.ing_cor, hogar.factor)
g_sin = gini_ponderado(hogar.ing_cor - hogar.transfer.fillna(0), hogar.factor)
print(f"Gini con transferencias : {g_con:.3f}   (publicado 0.391)")
print(f"Gini sin transferencias : {g_sin:.3f}   (publicado 0.450)")

Gini con transferencias : 0.401   (publicado 0.391)
Gini sin transferencias : 0.461   (publicado 0.450)


---
## 12. Tabla de referencia: la celda `entidad × tam_loc × est_socio`

Este es el artefacto que hace utilizable a la ENIGH en el pipeline geográfico.
Como la encuesta **no es representativa a nivel municipio**, la unidad de trabajo
correcta es la celda del diseño muestral. Cada celda se acompaña de su coeficiente
de variación y del semáforo de precisión del INEGI:

| CV | Interpretación |
|---|---|
| < 15 % | Precisión alta |
| 15 – 30 % | Precisión moderada |
| ≥ 30 % | Precisión baja: no usar |

El cálculo del error estándar usa **conglomerados últimos**: la varianza se estima
entre UPM dentro de estrato, no entre hogares. Ignorarlo subestima el error entre
1.1 y 1.9 veces, dado que el efecto de diseño por entidad va de 1.17 a 3.73.

In [36]:
def ee_conglomerados_ultimos(g, valor="ing_cor", peso="factor",
                             estrato="est_dis", upm="upm"):
    # Error estandar de una media ponderada bajo diseno estratificado por UPM.
    # Conglomerados ultimos + linealizacion de Taylor para el estimador de razon,
    # que es como el INEGI calcula las precisiones publicadas.
    d = g[[valor, peso, estrato, upm]].dropna()
    if d[upm].nunique() < 2:
        return np.nan
    w = d[peso].to_numpy(float); y = d[valor].to_numpy(float)
    Y = np.average(y, weights=w)
    # residual linealizado del estimador de razón
    d = d.assign(_z=w * (y - Y))
    por_upm = d.groupby([estrato, upm], as_index=False)["_z"].sum()
    var = 0.0
    for _, s in por_upm.groupby(estrato):
        k = len(s)
        if k < 2:
            continue
        var += k / (k - 1) * ((s["_z"] - s["_z"].mean()) ** 2).sum()
    return float(np.sqrt(var) / w.sum())


celdas = []
for (ent, tl, es), g in hogar.groupby(["cve_ent", "tam_loc", "est_socio"], dropna=False):
    media = media_ponderada(g, "ing_cor")
    ee = ee_conglomerados_ultimos(g)
    celdas.append({
        "cve_ent": ent, "tam_loc": tl, "est_socio": es,
        "n_hogares": len(g), "n_upm": g.upm.nunique(),
        "hogares_expandidos": g.factor.sum(),
        "ing_cor_medio": media, "ee": ee,
        "cv_%": 100 * ee / media if media and np.isfinite(ee) else np.nan,
    })
celdas = pd.DataFrame(celdas)
celdas["precision"] = pd.cut(celdas["cv_%"], [-1, 15, 30, np.inf],
                             labels=["Alta", "Moderada", "Baja"])

print(f"Celdas posibles     : 32 x 4 x 4 = 512")
print(f"Celdas observadas   : {len(celdas)}")
print(f"\nDistribución de precisión:")
display(celdas.precision.value_counts(dropna=False).to_frame("n_celdas"))
display(celdas.sort_values("cv_%").head(10))

Celdas posibles     : 32 x 4 x 4 = 512
Celdas observadas   : 427

Distribución de precisión:


,n_celdas
precision,
Alta,335
NaN,53
Moderada,32
Baja,7


,cve_ent,tam_loc,est_socio,n_hogares,n_upm,hogares_expandidos,ing_cor_medio,ee,cv_%,precision
310,24,1,1,10,2,4255,"41,800.14",0.00,0.00,Alta
198,16,2,1,49,3,38087,"56,899.52",0.00,0.00,Alta
192,15,4,3,22,2,33242,"98,004.44",0.00,0.00,Alta
248,19,3,4,10,2,10736,"134,001.10",0.00,0.00,Alta
189,15,3,4,14,3,49993,"111,925.23",0.00,0.00,Alta
83,07,2,3,27,2,17854,"56,754.72",0.00,0.00,Alta
278,21,3,3,23,2,28143,"67,259.55",0.00,0.00,Alta
20,02,3,1,43,2,20562,"65,093.87",0.00,0.00,Alta
153,13,2,4,9,2,5135,"62,122.53",0.00,0.00,Alta
221,17,3,4,7,2,4666,"139,747.08",0.00,0.00,Alta


In [37]:
# Cobertura práctica: qué proporción de la población vive en celdas usables
usable = celdas.precision.isin(["Alta", "Moderada"])
print(f"Celdas usables (CV < 30%)         : {usable.sum()} de {len(celdas)}")
print(f"Población cubierta por esas celdas: "
      f"{celdas.loc[usable,'hogares_expandidos'].sum()/celdas.hogares_expandidos.sum():.1%}")
print(f"\nCeldas con menos de 30 hogares    : {(celdas.n_hogares < 30).sum()}")
print(f"Celdas con menos de 10 UPM        : {(celdas.n_upm < 10).sum()}")

Celdas usables (CV < 30%)         : 367 de 427
Población cubierta por esas celdas: 98.6%

Celdas con menos de 30 hogares    : 106
Celdas con menos de 10 UPM        : 214


### Hallazgo 6 — cuántas celdas sobreviven al filtro de precisión

La proporción de celdas con CV bajo 30 % es la métrica que determina qué tanto
puede usarse la ENIGH como *prior* geográfico. Las celdas descartadas son
típicamente combinaciones raras: estrato alto en localidades rurales, estrato bajo
en zonas metropolitanas grandes.

Para esas celdas hay dos salidas: colapsar `est_socio` a dos niveles, o pasar a un
modelo jerárquico que tome prestada fuerza de las celdas vecinas (Fay-Herriot).
No hay una tercera: usar la celda ruidosa tal cual es propagar error al score.

---
## 13. Exportación

Formato parquet: preserva tipos, comprime bien y evita que `folioviv` vuelva a
perder los ceros a la izquierda al releer, que es exactamente el problema que
tuvimos que resolver al leer los CSV.

In [38]:
salidas = {
    "enigh2024_hogar":           hogar,
    "enigh2024_persona":         persona,
    "enigh2024_ingresos_clave":  ingresos.merge(clasif, on="clave", how="left"),
    "enigh2024_negocios":        negocios,
    "enigh2024_celdas_diseno":   celdas,
    "enigh2024_validacion":      validacion,
}

for nombre, df in salidas.items():
    ruta = RUTA_SALIDA / f"{nombre}.parquet"
    df.to_parquet(ruta, index=False, compression="snappy")
    print(f"  {nombre:<28} {len(df):>9,} filas x {df.shape[1]:>3} cols  "
          f"-> {ruta.stat().st_size/1e6:>7.1f} MB")

  enigh2024_hogar                 91,414 filas x 355 cols  ->    22.5 MB
  enigh2024_persona              308,598 filas x 303 cols  ->    25.6 MB
  enigh2024_ingresos_clave       391,563 filas x  23 cols  ->     7.0 MB
  enigh2024_negocios              36,888 filas x  13 cols  ->     0.9 MB
  enigh2024_celdas_diseno            427 filas x  10 cols  ->     0.0 MB
  enigh2024_validacion                10 filas x   5 cols  ->     0.0 MB


In [39]:
# Diccionario del dataset entregado: origen y significado de cada columna
def diccionario_final(df, nombre, esquema):
    base = esquema.drop_duplicates("variable").set_index("variable")
    filas = []
    for c in df.columns:
        raiz = (c.replace("hog_", "").replace("trab_", "")
                 .replace("_desc", "").replace("neg_", ""))
        filas.append({
            "dataset": nombre,
            "columna": c,
            "dtype": str(df[c].dtype),
            "pct_nulo": round(100 * df[c].isna().mean(), 1),
            "n_unicos": int(df[c].nunique(dropna=True)),
            "origen": ("hogar/vivienda" if c.startswith("hog_") else
                       "trabajo principal" if c.startswith("trab_") else
                       "negocios" if c.startswith("neg_") else
                       "etiqueta de catálogo" if c.endswith("_desc") else
                       "derivada" if raiz not in base.index else "microdato INEGI"),
            "etiqueta_inegi": base.loc[raiz, "etiqueta"] if raiz in base.index else "",
        })
    return pd.DataFrame(filas)

dicc_final = pd.concat([diccionario_final(hogar, "enigh2024_hogar", esquema),
                        diccionario_final(persona, "enigh2024_persona", esquema)],
                       ignore_index=True)
dicc_final.to_csv(RUTA_SALIDA / "enigh2024_diccionario_final.csv", index=False)
print(f"Diccionario final: {len(dicc_final):,} columnas documentadas")
display(dicc_final.groupby(["dataset", "origen"]).size().to_frame("n_columnas"))

Diccionario final: 658 columnas documentadas


n_columnas
dataset           origen                          
enigh2024_hogar   derivada                       7
                  etiqueta de catálogo           6
                  hogar/vivienda                 2
                  microdato INEGI              339
                  trabajo principal              1
enigh2024_persona derivada                      15
                  etiqueta de catálogo           5
                  hogar/vivienda                31
                  microdato INEGI              185
                  negocios                       6
                  trabajo principal             61

---
## 14. Resumen de la integración

Cambios de tamaño a lo largo del pipeline, para dejar constancia auditable.

In [40]:
resumen = pd.DataFrame([
    {"etapa": "1. CSV originales en disco", "unidad": "17 archivos",
     "filas": np.nan, "columnas": np.nan, "mb": inv.mb.sum()},
    {"etapa": "2. viviendas cargada", "unidad": "vivienda",
     "filas": len(viviendas), "columnas": viviendas.shape[1],
     "mb": viviendas.memory_usage(deep=True).sum()/1e6},
    {"etapa": "3. concentradohogar cargada", "unidad": "hogar",
     "filas": len(concentrado), "columnas": concentrado.shape[1],
     "mb": concentrado.memory_usage(deep=True).sum()/1e6},
    {"etapa": "4. poblacion cargada", "unidad": "persona",
     "filas": len(poblacion), "columnas": poblacion.shape[1],
     "mb": poblacion.memory_usage(deep=True).sum()/1e6},
    {"etapa": "5. ingresos (formato largo)", "unidad": "persona x clave",
     "filas": len(ingresos), "columnas": ingresos.shape[1],
     "mb": ingresos.memory_usage(deep=True).sum()/1e6},
    {"etapa": "6. ingresos pivoteados", "unidad": "persona",
     "filas": len(ing_persona), "columnas": ing_persona.shape[1],
     "mb": ing_persona.memory_usage(deep=True).sum()/1e6},
    {"etapa": "7. trabajos (todos)", "unidad": "persona x trabajo",
     "filas": len(trabajos), "columnas": trabajos.shape[1],
     "mb": trabajos.memory_usage(deep=True).sum()/1e6},
    {"etapa": "8. trabajo principal", "unidad": "persona ocupada",
     "filas": len(trabajo_ppal), "columnas": trabajo_ppal.shape[1],
     "mb": trabajo_ppal.memory_usage(deep=True).sum()/1e6},
    {"etapa": "9. TABLA HOGAR", "unidad": "hogar",
     "filas": len(hogar), "columnas": hogar.shape[1],
     "mb": hogar.memory_usage(deep=True).sum()/1e6},
    {"etapa": "10. TABLA PERSONA", "unidad": "persona",
     "filas": len(persona), "columnas": persona.shape[1],
     "mb": persona.memory_usage(deep=True).sum()/1e6},
])
display(resumen.style.format({"filas": "{:,.0f}", "columnas": "{:,.0f}", "mb": "{:,.1f}"}))

,etapa,unidad,filas,columnas,mb
0,1. CSV originales en disco,17 archivos,nan,nan,879.7
1,2. viviendas cargada,vivienda,"90,324",82,287.6
2,3. concentradohogar cargada,hogar,"91,414",126,132.6
3,4. poblacion cargada,persona,"308,598",185,"1,934.0"
4,5. ingresos (formato largo),persona x clave,"391,563",21,290.5
5,6. ingresos pivoteados,persona,"202,365",12,46.9
6,7. trabajos (todos),persona x trabajo,"164,325",60,395.1
7,8. trabajo principal,persona ocupada,"150,382",64,435.4
8,9. TABLA HOGAR,hogar,"91,414",355,821.4
9,10. TABLA PERSONA,persona,"308,598",303,"3,141.0"


---
## 15. Hallazgos consolidados

1. **La ENIGH no es una tabla, son cinco granos.** El entregable son dos tablas
   analíticas con grano verificado, no un CSV aplanado.

2. **`gastoshogar` es el 66 % del volumen y no se necesita.** `concentradohogar`
   ya trae los agregados de gasto trimestralizados. Saltarlo baja el pipeline de
   880 MB a ~300 MB de lectura.

3. **`folioviv` debe leerse como texto.** Es la falla silenciosa más común: al
   convertirse a entero pierde el cero inicial y todas las entidades 01-09 dejan
   de unir.

4. **`smg` viene dentro de `concentradohogar`.** El objetivo puede expresarse en
   salarios mínimos sin fuentes externas ni deflactores.

5. **`trabajos` trae SCIAN y SINCO a 4 dígitos.** Puente directo hacia DENUE
   (misma clasificación) e IMSS (tamaño de empresa). Es la variable de mayor valor
   del microdato para este proyecto.

6. **La celda `entidad × tam_loc × est_socio` es la unidad geográfica correcta.**
   512 celdas posibles; las que superan el filtro de CV son la base para bajar de
   entidad a AGEB sin inventar representatividad.

7. **Las variables de diseño no son opcionales.** `upm` + `est_dis` + `factor` son
   obligatorias para cualquier estimación. Un promedio ponderado solo por `factor`
   da el punto correcto pero errores estándar subestimados.

---

## 16. Siguientes pasos sugeridos

- **Ratio IMSS → ingreso del hogar.** Con `hogar`, estimar
  `ing_cor / trabajo` por decil, entidad y `est_socio`. Es la capa de corrección
  para la bandera objetivo construida desde IMSS.
- **Réplica de `est_socio` sobre Censo 2020 a nivel AGEB.** Los 42 indicadores
  del Anexo A del Diseño muestral son reproducibles; con eso cada solicitante
  recibe su celda.
- **Modelo de ingreso a nivel persona.** `persona` ya tiene el target
  (`ing_tri_total`, `ing_tri_sm`) y las covariables (edad, sexo, escolaridad,
  SCIAN, SINCO, `tam_emp`, prestaciones, `est_socio`, `tam_loc`). Estimar con
  pesos y validar con clusters por UPM, no con k-fold aleatorio.
- **Comparación con el modelo de bloques de ENOE.** Ambas encuestas comparten
  marco muestral y definición de `tam_loc`, así que los resultados son
  armonizables.